<a href="https://colab.research.google.com/github/lekhasree15/AI_LAB/blob/main/lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
def is_variable(x):

    return isinstance(x, str) and x.isidentifier() and x[0].islower()

def is_compound(x):
    return isinstance(x, (list, tuple)) and len(x) > 0

def occur_check(var, term, subst):
    if var == term:
        return True
    elif is_variable(term):
        if term in subst:
            return occur_check(var, subst[term], subst)
    elif is_compound(term):
        # Skip the operator (index 0) and check arguments
        return any(occur_check(var, arg, subst) for arg in term[1:])
    return False

def unify(x, y, subst=None):
    if subst is None:
        subst = {}

    # If substitution has already failed, pass it down
    if subst is None:
        return None

    # Case 1: Identical expressions
    if x == y:
        return subst

    # Case 2: x is a variable
    elif is_variable(x):
        return unify_var(x, y, subst)

    # Case 3: y is a variable
    elif is_variable(y):
        return unify_var(y, x, subst)

    # Case 4: Both are compound terms (e.g., ['f', 'x'] or ['P', 'x', 'y'])
    elif is_compound(x) and is_compound(y):
        # Must have the same function/predicate symbol and same arity
        if len(x) != len(y) or x[0] != y[0]:
            return None

        # Recursively unify arguments
        for arg_x, arg_y in zip(x[1:], y[1:]):
            subst = unify(arg_x, arg_y, subst)
            if subst is None:
                return None
        return subst

    # Case 5: Mismatch (e.g., constant vs function, or different constants)
    else:
        return None

def unify_var(var, val, subst):
    if var in subst:
        return unify(subst[var], val, subst)
    elif is_variable(val) and val in subst:
        return unify(var, subst[val], subst)
    elif occur_check(var, val, subst):
        # Occur check failed: prevent infinite loops (e.g., x unifies with f(x))
        return None
    else:
        # Successfully bind var to val
        new_subst = subst.copy()
        new_subst[var] = val
        return new_subst

def run_all_cases():
    test_cases = [
        # --- SUCCESSFUL CASES ---
        {
            "desc": "Successful Unification: Identical constants",
            "x": "A", "y": "A"
        },
        {
            "desc": "Successful Unification: Variable and Constant",
            "x": "x", "y": "A"
        },
        {
            "desc": "Successful Unification: Variable and Variable",
            "x": "x", "y": "y"
        },
        {
            "desc": "Successful Unification: Predicate arguments mapping",
            "x": ["P", "x", "B"],
            "y": ["P", "A", "y"]
        },
        {
            "desc": "Successful Unification: Transitive bindings",
            "x": ["P", "x", "y"],
            "y": ["P", "y", "A"]
        },
        {
            "desc": "Successful Unification: Complex structures with functions",
            "x": ["P", "x", ["f", "y"]],
            "y": ["P", ["g", "z"], ["f", "B"]]
        },

        # --- FAIL CASES ---
        {
            "desc": "Fail: Constant mismatch",
            "x": "A", "y": "B"
        },
        {
            "desc": "Fail: Predicate/Function name mismatch",
            "x": ["P", "x"],
            "y": ["Q", "x"]
        },
        {
            "desc": "Fail: Arity (argument count) mismatch",
            "x": ["P", "x"],
            "y": ["P", "x", "y"]
        },
        {
            "desc": "Fail: Occur Check (Variable inside itself)",
            "x": "x",
            "y": ["f", "x"]
        },
        {
            "desc": "Fail: Indirect Occur Check through bindings",
            "x": ["P", "x", "y"],
            "y": ["P", ["f", "y"], "x"]
        }
    ]

    for i, tc in enumerate(test_cases, 1):
        result = unify(tc["x"], tc["y"])
        status = "SUCCESS" if result is not None else "FAILED"
        print(f"Test Case {i}: {tc['desc']}")
        print(f"  Expression 1: {tc['x']}")
        print(f"  Expression 2: {tc['y']}")
        print(f"  Result:       {status} -> {result}")
        print("-" * 60)

run_all_cases()

Test Case 1: Successful Unification: Identical constants
  Expression 1: A
  Expression 2: A
  Result:       SUCCESS -> {}
------------------------------------------------------------
Test Case 2: Successful Unification: Variable and Constant
  Expression 1: x
  Expression 2: A
  Result:       SUCCESS -> {'x': 'A'}
------------------------------------------------------------
Test Case 3: Successful Unification: Variable and Variable
  Expression 1: x
  Expression 2: y
  Result:       SUCCESS -> {'x': 'y'}
------------------------------------------------------------
Test Case 4: Successful Unification: Predicate arguments mapping
  Expression 1: ['P', 'x', 'B']
  Expression 2: ['P', 'A', 'y']
  Result:       SUCCESS -> {'x': 'A', 'y': 'B'}
------------------------------------------------------------
Test Case 5: Successful Unification: Transitive bindings
  Expression 1: ['P', 'x', 'y']
  Expression 2: ['P', 'y', 'A']
  Result:       SUCCESS -> {'x': 'y', 'y': 'A'}
---------------------